# UK Energy Research Centre Hackathon 2026
## Track 02: Energy Poverty & Equity Starter Notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/0ladayo/UK-Energy-Research-Centre-Hackathon/blob/main/notebooks/track_02_energy_poverty_equity_starter.ipynb)

### Challenge Focus:
Investigate geographic, socio-economic, and dwelling-level disparities in energy poverty, energy consumption, and efficiency across England & Wales.
- **Key Themes**: Fuel poverty prevalence (LILEE indicator), deprivation deciles (IMD 2019), domestic gas/electricity usage at LSOA level, and household consumption drivers (NEED framework).
- **Data Dictionary**: Refer to `DATASET_DICTIONARY.md` for full schema, field descriptions, and units.


### 1. Environment Setup & Data Loading
If running in **Google Colab**, this cell automatically clones the repository and navigates into the workspace directory.


In [ ]:
import sys
import os

# Detect if running in Google Colab
if 'google.colab' in sys.modules:
    print("Running in Google Colab environment...")
    !git clone https://github.com/0ladayo/UK-Energy-Research-Centre-Hackathon.git
    %cd UK-Energy-Research-Centre-Hackathon
else:
    # Ensure current directory is repository root if running locally
    if os.path.exists("Energy Poverty & Equity"):
        print("Running locally from repository root.")
    elif os.path.exists("../Energy Poverty & Equity"):
        os.chdir("..")
        print(f"Changed working directory to: {os.getcwd()}")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("Libraries imported successfully!")


### 2. Loading Track 02 Datasets
We load:
1. **Sub-regional Fuel Poverty (2024 / 2022 stats)**: LSOA-level fuel poor households & LILEE metrics.
2. **Index of Multiple Deprivation (IMD 2019)**: Socio-economic deprivation scores and deciles.
3. **National Energy Efficiency Data-Framework (NEED 50k sample)**: Dwelling age, property type, floor area, gas & elec consumption.
4. **LSOA Domestic Gas & Electricity Consumption**: Meter counts, median, and mean consumption.


In [ ]:
data_dir = "Energy Poverty & Equity"

# 1. Fuel Poverty Sub-Regional (Read Sheet: 'Table 2')
fuel_poverty_path = os.path.join(data_dir, "fuel-poverty-sub-regional-2026-2024-data-tables.xlsx")
# Read Table 2 where LSOA data resides (header is typically row 2)
df_fuel_poverty = pd.read_excel(fuel_poverty_path, sheet_name="Table 2", header=2)
print("Fuel Poverty Data Shape:", df_fuel_poverty.shape)
display(df_fuel_poverty.head(3))


In [ ]:
# 2. Index of Multiple Deprivation (IMD 2019)
imd_path = os.path.join(data_dir, "File_1_-_IMD2019_Index_of_Multiple_Deprivation.xlsx")
df_imd = pd.read_excel(imd_path, sheet_name=0)
print("IMD 2019 Shape:", df_imd.shape)
display(df_imd.head(3))


In [ ]:
# 3. National Energy Efficiency Data-Framework (NEED 50k Sample)
need_path = os.path.join(data_dir, "anonymised-NEED-data-2024-50k.csv")
df_need = pd.read_csv(need_path)
print("NEED 50k Sample Shape:", df_need.shape)
display(df_need.head(3))


In [ ]:
# 4. LSOA Domestic Electricity & Gas Consumption (2024 releases)
elec_path = os.path.join(data_dir, "lso_domestic_elec_2024.xlsx")
gas_path = os.path.join(data_dir, "lsoa_domestic_gas_2024.xlsx")

df_elec = pd.read_excel(elec_path, sheet_name=0, header=4)
df_gas = pd.read_excel(gas_path, sheet_name=0, header=4)

print("LSOA Electricity Shape:", df_elec.shape)
print("LSOA Gas Shape:", df_gas.shape)


### 3. Exploratory Analysis & Starter Merges
#### Question 1: How does Fuel Poverty correlate with Deprivation (IMD Decile)?


In [ ]:
# Clean column names for Fuel Poverty and IMD
fp_lsoa_col = [c for c in df_fuel_poverty.columns if "LSOA Code" in str(c) or "LSOA code" in str(c)][0]
fp_pct_col = [c for c in df_fuel_poverty.columns if "Proportion" in str(c) or "%" in str(c) or "proportion" in str(c)][0]

imd_lsoa_col = [c for c in df_imd.columns if "LSOA code" in str(c)][0]
imd_decile_col = [c for c in df_imd.columns if "Decile" in str(c)][0]

# Merge on LSOA Code
merged_fp_imd = pd.merge(
    df_fuel_poverty[[fp_lsoa_col, fp_pct_col]],
    df_imd[[imd_lsoa_col, imd_decile_col]],
    left_on=fp_lsoa_col,
    right_on=imd_lsoa_col,
    how="inner"
)

# Group average fuel poverty by IMD Deprivation Decile (1 = Most Deprived, 10 = Least Deprived)
avg_fp_by_decile = merged_fp_imd.groupby(imd_decile_col)[fp_pct_col].mean()

plt.figure(figsize=(10, 5))
avg_fp_by_decile.plot(kind="bar", color="#d9534f", edgecolor="black")
plt.title("Average Fuel Poverty Proportion by IMD Deprivation Decile (1 = Most Deprived)")
plt.xlabel("IMD Deprivation Decile")
plt.ylabel("Avg % of Fuel Poor Households")
plt.grid(axis="y", linestyle="--", alpha=0.7)
plt.show()


#### Question 2: What property archetypes drive the highest energy consumption in the NEED dataset?


In [ ]:
# Inspect electricity & gas consumption distributions by property type or band
display(df_need.describe())

# Check columns available in NEED sample
print("NEED Columns:", df_need.columns.tolist()[:10])


### 4. Hackathon Starter Ideas & Challenge Questions
- **Spatial Hotspot Identification**: Use the included LSOA GeoJSON boundary file (`Lower_layer_Super_Output_Areas_...geojson`) with `geopandas` to map the most acute fuel poverty clusters in England & Wales.
- **Predictive Modeling**: Train a machine learning model to predict high fuel poverty risk areas based on gas consumption, electricity consumption, and housing characteristics.
- **Policy Intervention Targeting**: Formulate an intervention policy ranking which local authorities or LSOAs should receive priority funding for home retrofit insulation.
